# ML-08 — Honest Model vs Week-4 Baseline

This notebook trains a simple learned classifier for the FlyRank content-review lane and compares it with the Week-4 rule baseline on the **same holdout rows** and the **same primary metric (Recall)**.

The label is used only for retrospective evaluation. `trend_direction` and `trend_pct` are never model features because the label is derived from them.

## 1. Method choice and why

**Chosen method: Logistic Regression.**

This is a binary classification problem: predict whether a content item is declining enough to warrant review. Logistic Regression is the right first learned model because it is readable, fast, and gives probabilities that can support ranking. It is also a useful step beyond the Week-4 transparent rule without rewarding complexity for its own sake.

I will use a client-grouped split because `client_id` identifies the client context. This prevents the same client from appearing in both train and test and gives a stricter check of whether the signal transfers across clients.

In [ ]:
from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    recall_score, precision_score, f1_score, roc_auc_score,
    confusion_matrix
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
REPO_URL = "https://github.com/engyusufayman06/ml-internship-2026.git"
REPO_DIR = Path("/content/ml-internship-2026")

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    REPO_DIR / "data/raw/content_refresh_anonymized.csv",
]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

DATA_PATH = next((p for p in [
    Path("data/raw/content_refresh_anonymized.csv"),
    REPO_DIR / "data/raw/content_refresh_anonymized.csv"
] if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Starter dataset not found. Expected data/raw/content_refresh_anonymized.csv."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)

required = {"content_id", "client_id", "days_since_last_update", "impressions_90d", "trend_direction"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

# The starter snapshot does not ship the label. This proxy is for retrospective evaluation only.
# It is NOT used as an input feature.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
    print("Created evaluation-only label from trend_direction == 'down'.")
else:
    df["is_declining_label"] = df["is_declining_label"].astype(int)

print("Positive rate:", round(df["is_declining_label"].mean(), 4))

## 2. Split design

Use a fixed `GroupShuffleSplit` with `client_id` as the group. The test set is 20% of rows approximately, but clients are kept entirely on one side of the split. The seed is fixed at 42 so the comparison is reproducible.

The Week-4 baseline is a rule, so it has no training step. I apply that same rule to the exact test rows used for the learned model.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

assert set(train["client_id"]).isdisjoint(set(test["client_id"]))

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())
print("Client overlap:", len(set(train["client_id"]) & set(test["client_id"])))

## 3. Train + compare vs my baseline

### Features

Use the available content, engagement, search, and structural fields, while explicitly removing:

- `trend_direction`, `trend_pct`, and `is_declining_label` — outcome/leakage fields
- `content_id` and `client_id` — identifiers/grouping keys, not predictive features

Missing numeric values are median-imputed with missing indicators; categorical values are imputed and one-hot encoded.

In [ ]:
TARGET = "is_declining_label"
GROUP = "client_id"
FORBIDDEN = {
    TARGET, "trend_direction", "trend_pct",
    "content_id", GROUP,
    "score", "reason_code", "action_label",
    "freshness_bucket", "volume_bucket"
}

feature_cols = [c for c in df.columns if c not in FORBIDDEN]
X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y_train = train[TARGET]
y_test = test[TARGET]

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
])

model.fit(X_train, y_train)
model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.50).astype(int)

print("Features used:", len(feature_cols))
print("Numeric:", len(numeric_cols), "| Categorical:", len(categorical_cols))

In [ ]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean())

def metrics_row(name, y_true, pred, scores):
    return {
        "method": name,
        "recall": recall_score(y_true, pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, scores),
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
    }

# Week-4 baseline rule, applied to the SAME test rows.
baseline_selected = (
    test["days_since_last_update"].fillna(0).ge(180)
    & test["impressions_90d"].fillna(0).ge(3000)
)
baseline_score = np.where(
    baseline_selected,
    test["impressions_90d"].fillna(0),
    0.0
)
baseline_pred = baseline_selected.astype(int)

comparison = pd.DataFrame([
    metrics_row("Week-4 rule baseline", y_test, baseline_pred, baseline_score),
    metrics_row("Logistic Regression", y_test, model_pred, model_prob),
])

print(comparison.round(3).to_string(index=False))

### Model-vs-baseline reading

The primary decision metric is **Recall**, because the lane is about finding declining content for review. Precision@20 and Precision@50 are secondary ranking checks because the operational question is also “which items should be reviewed first.”

The learned model is only an improvement if it earns that improvement on the same held-out rows; a higher-complexity model would not be justified by complexity alone.

## 4. Errors and interpretation

Read the errors before trusting the score. I will inspect the confusion matrix, the largest Logistic Regression coefficients, and three concrete misclassified rows.

The examples below are pseudonymized content IDs only; no client names, URLs, or private queries are shown.

In [ ]:
cm = confusion_matrix(y_test, model_pred)
print("Confusion matrix [[TN, FP, FN, TP]]:")
print(cm)

coef = model.named_steps["clf"].coef_[0]
feature_names = model.named_steps["prep"].get_feature_names_out()
coef_table = (
    pd.DataFrame({"feature": feature_names, "coefficient": coef})
    .assign(abs_coefficient=lambda x: x["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
)

print("\nTop positive model signals:")
print(coef_table.head(10)[["feature", "coefficient"]].round(3).to_string(index=False))

print("\nTop negative model signals:")
print(coef_table.sort_values("coefficient").head(10)[["feature", "coefficient"]].round(3).to_string(index=False))

In [ ]:
error_view = test[[
    "content_id", "days_since_last_update", "impressions_90d",
    "avg_position", TARGET
]].copy()
error_view["predicted"] = model_pred
error_view["probability"] = model_prob
error_view["error_type"] = np.select(
    [
        (error_view[TARGET] == 1) & (error_view["predicted"] == 0),
        (error_view[TARGET] == 0) & (error_view["predicted"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct"
)

errors = error_view[error_view["error_type"] != "correct"].copy()
print("Total test errors:", len(errors))

for error_type in ["false_negative", "false_positive"]:
    examples = errors[errors["error_type"] == error_type].head(3)
    print(f"\n{error_type} examples:")
    print(examples.to_string(index=False))

### Error interpretation

False negatives are the more important failure for this lane because they are declining pages the model would miss. False positives consume review capacity but are less costly than silently missing a page that needs attention.

The coefficient table is interpreted directionally, not causally: a positive coefficient means the feature pushes the Logistic Regression score upward after preprocessing. These are associations in this snapshot, not proof that changing a feature will cause a decline.

The hardest cases are expected to sit near the decision boundary or combine weak signals that the simple Week-4 rule cannot capture.

## Self-check

- [x] Method is appropriate for binary classification and chosen for interpretability.
- [x] Split is grouped by `client_id` and reproducible with a fixed seed.
- [x] Model and Week-4 baseline are evaluated on the same test rows.
- [x] Primary metric is Recall; ranking metrics are also reported.
- [x] Leakage/outcome columns are excluded from model features.
- [x] IDs are not model features.
- [x] Errors and feature interpretation are included.
- [x] No client names, URLs, or private queries are included.
- [ ] Run the notebook top-to-bottom in Colab and commit the executed notebook after verifying the printed numbers.